# TextSwinUMamba — Colab Training

Trains `TextSwinUMambaD` on ISIC 2017 with GPT-4o captions as text input.

**Layout:**
- This **code repo** is cloned from GitHub on every session (cheap, ~seconds).
- **Artifacts** (checkpoints, logs, dataset, pretrained weights, text-feature cache)
  live on Google Drive and survive disconnects.

**Expected Drive layout** (set up once):
```
MyDrive/TextSwinUMamba/
├── datasets/isic2017/{train,val}/{images,masks}/
├── captions/captions.jsonl
├── pretrained/vmamba_tiny_e292.pth
├── cache/text_features.pt     # auto-generated by step 5
└── runs/                       # checkpoints + TensorBoard logs land here
```

**Resume:** if Colab disconnects, re-run the notebook top to bottom. Step 8 calls
`train.py --resume auto`, which picks up from `last.pth` on Drive.

## 1. Configure remote

Edit this URL once, then run all cells.

In [ ]:
TEXTSWINUMAMBA_REPO = 'https://github.com/<you>/TextSwinUMamba.git'
TEXTSWINUMAMBA_BRANCH = 'main'

## 2. Mount Drive (artifacts only)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/TextSwinUMamba'
DATA_ROOT  = f'{DRIVE_ROOT}/datasets/isic2017'
CAPTIONS   = f'{DRIVE_ROOT}/captions/captions.jsonl'
PRETRAINED = f'{DRIVE_ROOT}/pretrained/vmamba_tiny_e292.pth'
TEXT_CACHE = f'{DRIVE_ROOT}/cache/text_features.pt'
RUNS_DIR   = f'{DRIVE_ROOT}/runs'

import os
for p in [DRIVE_ROOT, RUNS_DIR, os.path.dirname(TEXT_CACHE)]:
    os.makedirs(p, exist_ok=True)

# Quick sanity check
for label, path in [('dataset', DATA_ROOT), ('captions', CAPTIONS), ('pretrained', PRETRAINED)]:
    print(f'{label:11s}: {path}  [{"OK" if os.path.exists(path) else "MISSING"}]')

## 3. Clone the repo

The upstream Swin-UMamba network code is already committed at
`models/swin_umamba_d.py` under Apache 2.0, so we only need this repo.

In [ ]:
%cd /content
!rm -rf TextSwinUMamba
!git clone --branch {TEXTSWINUMAMBA_BRANCH} {TEXTSWINUMAMBA_REPO} TextSwinUMamba
%cd /content/TextSwinUMamba
!ls

## 4. Install dependencies

`mamba-ssm` and `causal-conv1d` are the only tricky ones; pin versions if Colab's
default torch / CUDA combination breaks the install.

In [ ]:
!nvidia-smi
!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda)"

In [ ]:
!pip install -q --upgrade pip
!pip install -q causal-conv1d>=1.2 mamba-ssm>=1.2
!pip install -q -r requirements.txt

## 5. Precompute text features + sanity check captions

The text-feature cache lives on Drive and survives across sessions.

In [ ]:
# Sanity check: do all images have captions?
!python scripts/verify_caption_coverage.py --isic_root {DATA_ROOT} --captions {CAPTIONS}

In [ ]:
# Precompute BERT features for all captions. Cached on Drive; skipped if it already exists.
import os
if not os.path.exists(TEXT_CACHE):
    !python scripts/precompute_text_features.py \
        --captions {CAPTIONS} \
        --out {TEXT_CACHE} \
        --model bert-base-uncased \
        --pool attention \
        --max_length 64
else:
    print('Text features already cached at', TEXT_CACHE)

## 6. Patch config to point at Drive paths

Writes a Colab-flavored copy alongside the canonical one — no edits committed back to git.

In [ ]:
import yaml
src = 'configs/isic2017.yaml'
dst = 'configs/isic2017_colab.yaml'
with open(src) as f:
    cfg = yaml.safe_load(f)
cfg['data']['isic_root']           = DATA_ROOT
cfg['data']['captions_jsonl']      = CAPTIONS
cfg['data']['text_features_cache'] = TEXT_CACHE
cfg['model']['pretrained_ckpt']    = PRETRAINED
cfg['output']['base_dir']          = RUNS_DIR
cfg['output']['tensorboard']       = True
cfg['train']['max_hours']          = 11.5   # graceful stop before Colab kicks us
with open(dst, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print('wrote', dst)
print('  run_dir will be:', f"{RUNS_DIR}/{cfg['run_name']}")

## 7. Launch TensorBoard

Points at the Drive runs folder, so the same TensorBoard pane keeps working
across disconnects.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {RUNS_DIR} --port 6006

## 8. Train (auto-resumes from `last.pth` on Drive)

If the runtime disconnects mid-training, re-mount Drive, re-run cells 3–6, then re-run
this single cell. Training picks up at the next epoch with optimizer / scheduler /
RNG state restored.

In [ ]:
!python train.py --config configs/isic2017_colab.yaml --resume auto

## 9. Evaluate the best checkpoint

In [ ]:
RUN_NAME = yaml.safe_load(open('configs/isic2017_colab.yaml'))['run_name']
BEST_CKPT = f'{RUNS_DIR}/{RUN_NAME}/best.pth'
!python evaluate.py --config configs/isic2017_colab.yaml --ckpt {BEST_CKPT}